# 03 — Tagging blocks

**Purpose.** Cut every usable Item 1 into document-order blocks of at most
`BLOCK_SENTENCES` sentences — the context unit of every Layer-2 tagging call
(04). Document order, not similarity order: which sentences the model reads
together is decided by the filing itself, and no Layer-1 output touches the
sampling frame.

**Every filing-year is tagged directly** (decision 2026-08-22). A regime
collapse — merge consecutive filings whose normalized sentence sets overlap,
tag one representative — was built, measured, and rejected: within-firm
year-over-year Jaccard has median 0.41 (firms re-word annually), so at a 0.6
floor the collapse bought a 16% call reduction at the price of an
indirection between hearings and judged text. Tagging runs once; the yearly
grid keeps the INV-25 join trivial: a hearing reads the firm's most recent
10-K **filed before** the hearing date, and that filing's own tags. The
measured rejection lives in this notebook's git history.

**Inputs.** `item1/sentences.parquet` (01).

**Outputs.** `interim/relevance/tagging/blocks.parquet` — one row per
tagging block: accessionNo, block_idx, `sent_idx` range.

**Gate.** Blocks partition every filing's sentences exactly; call counts
printed.

## Setup

In [1]:
import sys

sys.path.insert(0, "../lib")

import numpy as np
import pandas as pd

from paths import relevance_run_dir
from tagging import BLOCK_SENTENCES

ITEM1 = relevance_run_dir("item1")
OUT = relevance_run_dir("tagging")

sentences = pd.read_parquet(ITEM1 / "sentences.parquet",
                            columns=["accessionNo", "gvkey", "fyear", "sent_idx"])
print(f"filings   : {sentences['accessionNo'].nunique():,}")
print(f"firms     : {sentences['gvkey'].nunique():,}")
print(f"sentences : {len(sentences):,}")
print(f"block size: {BLOCK_SENTENCES} sentences")

filings   : 21,966
firms     : 1,061
sentences : 6,306,793
block size: 40 sentences


## Cut blocks

Consecutive `sent_idx` runs per filing — the exact rows the tagging calls
number, so a cited index resolves to one table row forever.

In [2]:
rows = []
for acc, grp in sentences.groupby("accessionNo", sort=False):
    si = grp["sent_idx"].to_numpy()
    assert (np.diff(si) == 1).all() and si[0] == 0, "sent_idx not contiguous"
    for b in range(0, len(si), BLOCK_SENTENCES):
        chunk = si[b:b + BLOCK_SENTENCES]
        rows.append({"accessionNo": acc,
                     "block_idx": b // BLOCK_SENTENCES,
                     "sent_lo": int(chunk[0]), "sent_hi": int(chunk[-1]),
                     "n_sentences": int(len(chunk))})
blocks = pd.DataFrame(rows)

per_filing = blocks.groupby("accessionNo").size()
print(f"tagging blocks    : {len(blocks):,}")
print(f"blocks per filing : median {per_filing.median():.0f}, "
      f"p90 {per_filing.quantile(0.9):.0f}, max {per_filing.max():,}")
print(f"\nPass-A calls = {len(blocks):,}; Pass-B calls = blocks x themes "
      f"flagged there (measured on 04's dev read before any full run)")

tagging blocks    : 168,333
blocks per filing : median 6, p90 14, max 107

Pass-A calls = 168,333; Pass-B calls = blocks x themes flagged there (measured on 04's dev read before any full run)


## Write and verify

In [3]:
blocks.to_parquet(OUT / "blocks.parquet", index=False)
p = OUT / "blocks.parquet"
print(f"wrote {p}  ({p.stat().st_size / 1e6:.1f} MB)")

bl = pd.read_parquet(p)
assert int(bl["n_sentences"].sum()) == len(sentences), \
    "blocks do not partition the sentences"
assert not bl.duplicated(["accessionNo", "block_idx"]).any()
assert (bl["sent_hi"] - bl["sent_lo"] + 1 == bl["n_sentences"]).all()
print(f"1) blocks partition all {len(sentences):,} sentences, keys unique")
print("\nDone. Next: 04 (tagging promptbook — dev sample, researcher read).")

wrote /scratch2/acroh/projects/llm_method/data/interim/relevance/tagging/blocks.parquet  (0.6 MB)
1) blocks partition all 6,306,793 sentences, keys unique

Done. Next: 04 (tagging promptbook — dev sample, researcher read).
